**Load and inspect the dataset**

In [2]:
import pandas as pd
from google.colab import files


# Read the CSV
df = pd.read_csv("LSNM2024_150K_ML_ready.csv")
print("Shape (rows, columns):", df.shape)

display(df.head())

# If any missing values. We expect 0.
print("\nMissing values:", df.isna().sum().sum())

print("\nClass balance:")
print(df["label"].value_counts())

print("\nNon-numeric columns:", df.select_dtypes(exclude="number").columns.tolist())

Shape (rows, columns): (150000, 46)


,frame_length,ip_length,ip_ttl,ip_flags,ip_frag_offset,ip_dscp,ip_version,tcp_length,tcp_window_size,tcp_flags,...,http_has_sql_keyword,http_has_script_tag,http_has_cookie,http_has_user_agent,dns_query_length,dns_query_type_code,ethernet_type_code,protocol_code,attack_type,label
0,66,52.0,64.0,64,0.0,0,4.0,0.0,11584.0,16,...,0,0,0,0,0,0.0,1,1,SSH-BruteForce,1
1,74,60.0,64.0,64,0.0,0,4.0,0.0,5792.0,18,...,0,0,0,0,0,0.0,1,1,Fuzzing,1
2,11650,11636.0,64.0,64,0.0,0,4.0,11584.0,5824.0,16,...,0,0,0,0,0,0.0,1,5,Benign,0
3,66,52.0,64.0,64,0.0,0,4.0,0.0,5824.0,16,...,0,0,0,0,0,0.0,1,1,Benign,0
4,66,52.0,64.0,64,0.0,0,4.0,0.0,5824.0,17,...,0,0,0,0,0,0.0,1,1,Benign,0



Missing values: 0

Class balance:
label
1    75000
0    75000
Name: count, dtype: int64

Non-numeric columns: ['attack_type']


**Split into training and testing sets (80/20)**

In [3]:
from sklearn.model_selection import train_test_split

# X = the inputs (features the model learns from)
X = df.drop(columns=["attack_type", "label"])

# y = the answer to predict
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=df["attack_type"] # same split every time
)

print("Training set:", X_train.shape)
print("Testing set: ", X_test.shape)

Training set: (120000, 44)
Testing set:  (30000, 44)


**Train Algorithm 1 (Random Forest)**

In [4]:
import time
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,   # number of trees in the forest
    n_jobs=-1,          # use all available CPU cores (faster)
    random_state=42     # makes results repeatable
)

# Time the training so we can compare speed later
start = time.time()
rf_model.fit(X_train, y_train)      # actual "learning" step
rf_time = time.time() - start

print(f"Random Forest trained in {rf_time:.1f} seconds")

Random Forest trained in 19.5 seconds


**Train Algorithm 2 (LightGBM)**

In [5]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,     # number of boosting rounds (trees)
    learning_rate=0.1,    # how big a step each new tree takes
    random_state=42,
    verbose=-1            # keeps the output quiet
)

start = time.time()
lgbm_model.fit(X_train, y_train)
lgbm_time = time.time() - start

print(f"LightGBM trained in {lgbm_time:.1f} seconds")

LightGBM trained in 3.6 seconds


**Evaluating & Comparing Both Models**

In [8]:
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

models = {"Random Forest": rf_model, "LightGBM": lgbm_model}
train_times = {"Random Forest": rf_time, "LightGBM": lgbm_time}

results, conf_mats = [], {}

for name, model in models.items():
    y_pred = model.predict(X_test)               # predicted class: 0 or 1
    y_prob = model.predict_proba(X_test)[:, 1]   # probability of "Attack"

    results.append({
        "Model": name,
        "Accuracy (%)":  round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred) * 100, 2),
        "Recall (%)":    round(recall_score(y_test, y_pred) * 100, 2),
        "F1 Score (%)":  round(f1_score(y_test, y_pred) * 100, 2),
        "ROC-AUC (%)":   round(roc_auc_score(y_test, y_prob) * 100, 2),
        "Train Time (s)": train_times[name]
    })
    conf_mats[name] = confusion_matrix(y_test, y_pred)

# ---- Comparison table ----
results_df = pd.DataFrame(results).set_index("Model")
print("=== MODEL COMPARISON (Percentages) ===")
print(results_df)

# ---- Confusion matrices (numeric only) ----
print("\n=== CONFUSION MATRICES ===")
for name, cm in conf_mats.items():
    print(f"\n{name}:\n{cm}")


=== MODEL COMPARISON (Percentages) ===
               Accuracy (%)  Precision (%)  Recall (%)  F1 Score (%)  \
Model                                                                  
Random Forest         94.54          94.66       94.39         94.53   
LightGBM              96.32          98.36       94.21         96.24   

               ROC-AUC (%)  Train Time (s)  
Model                                       
Random Forest        98.68       19.497664  
LightGBM             99.34        3.636182  

=== CONFUSION MATRICES ===

Random Forest:
[[14202   798]
 [  841 14159]]

LightGBM:
[[14764   236]
 [  869 14131]]


**Save the models as .pkl files**

In [9]:
import joblib
from google.colab import files

joblib.dump(rf_model,   "random_forest_model.pkl", compress=3)
joblib.dump(lgbm_model, "lightgbm_model.pkl")

joblib.dump(list(X.columns), "feature_columns.pkl")

rf_loaded = joblib.load("random_forest_model.pkl")
print("Reloaded model accuracy:", (rf_loaded.predict(X_test) == y_test).mean())

files.download("random_forest_model.pkl")
files.download("lightgbm_model.pkl")
files.download("feature_columns.pkl")

Reloaded model accuracy: 0.9453666666666667


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>